# PE_V2X_Reliability Project Report (Approach A)

This report presents, in a systematic way, the research motivation, the three-scenario modeling, the mechanism and metric definitions, the reproducibility design, as well as a structured interpretation and discussion of Fig01–Fig07. Fig01–Fig07 are shown in the main text as PNG previews (rendered from the authoritative PDF).

**Date:** 2026-03-01


## 1 Research motivation and problem definition

The core requirement of V2X safety messages is not “maximum bandwidth” or “minimum average latency”. Instead, under **highly dynamic topology**, **complex propagation environments**, and **contention on a shared wireless channel**, the system must continuously satisfy the **reliability and timeliness constraints** required by cooperative perception and cooperative decision-making. In real road deployments, degraded environments (urban blockage, tunnels) and peak-hour congestion are not rare “extreme cases”; they are the normal boundary conditions that engineering systems must face: urban street canyons, interchanges, and long tunnels are widespread, and traffic peaks commonly increase channel busy ratio. Therefore, building an **explainable, reproducible, and extensible** evaluation framework around “V2X reliability and timeliness under degraded environments” has clear engineering significance and research value.

From a service perspective, the failure modes of V2X safety messages should be distinguished at least into two categories, because they affect safety applications through different pathways:

1) **Reliability failure (Non-delivery)**: the packet does not arrive, reflected by a decrease of PDR (Packet Delivery Ratio).  
2) **Timeliness failure (Late delivery)**: the packet arrives but exceeds the deadline; for cooperative decision-making it is effectively invalid, i.e., an “implicit failure”.

This implies that a single metric (e.g., average delay or a single PDR) is insufficient for rigorous conclusions: averages often hide **tail risk (tail latency)**; and with deadlines, a right-shift in the tail distribution directly converts into late deliveries, reducing the effective success probability of the service. To avoid “phenomenological descriptions without attribution”, this project adopts a system-level modeling approach with **mechanistic interpretability**, and reports under a unified protocol:

- **Reliability decay with distance**: PDR vs distance;  
- **Delay distribution and tail risk**: Delay CDF, p95/p99;  
- **Deadline-aware effective success**: `success_phy` (PHY success), `late` (deadline miss), `success` (timely success).

Meanwhile, the degraded environment is consolidated into three representative scenarios (RefPlus / UrbMaskPlus / TunnelPlus), and a **NoCong / Cong** two-regime contrast is introduced. With other protocol elements frozen, congestion effects (contention and collision) are injected only via a congestion switch. This enables answering two questions closer to “system design decisions”:

- Does peak-hour congestion become the common dominant factor, compressing environmental differences and changing strategy selection?  
- With retransmissions `ret=0/1/2` as the primary enhancement lever, how do the benefits (reliability improvement) and costs (heavier tail delay, higher late rate) trade off under different scenarios/regimes?

---

## 2 Project build-up and overall pipeline

The outcomes of this project can be summarized as a **system-level evaluation pipeline**: input generation → event simulation → aggregated statistics → figure production. It is solidified as a reproducible experiment package via a **run_id output contract + command snapshots**. The goal is not “to obtain one set of results”, but to establish a reusable skeleton so that future extensions (more degraded environments, RSU, strategy improvements, real-trajectory inputs) can still keep the same metric definitions and evidence chain.

### 2.1 Input generation layer: explicit and auditable scenario & traffic inputs

To avoid “key settings hidden in code leading to irreproducibility”, inputs are decomposed and frozen into traceable files/configuration:

- **Mobility input (trajectories)**: RefPlus geometry + IDM car-following + signal phases + cross/turn mechanisms, generating trajectory CSV.  
- **Building input (UrbMask)**: street-block geometry expressed by rectangular building blocks, generating building CSV (footprint, height, distribution pattern, etc.).  
- **Tunnel input (Tunnel)**: tunnel interval and impairment parameters, generating tunnel config CSV.

The engineering value of this layer is that reviewers do not need the author’s local directory or to understand every line of code; they can reproduce the same simulation conditions from the input files and command snapshots.

### 2.2 Event simulation layer: packet-level samples as the basic unit

The simulation core treats each safety message as a basic sample and outputs three types of information:

- **Geometry/state variables**: distance, link midpoint position, scenario state (e.g., NLOS / inside tunnel), etc.;  
- **Outcome variables**: `success_phy`, `delay_ms`, `late`, `success`;  
- **Evidence fields**: blockage intensity (e.g., `d_min`/`b`), tunnel position (`tunnel_u`), congestion proxies (`n_cs`/CBR/`p_col`/`cong_delay_ms`), etc.

Evidence fields ensure that subsequent figures can not only show “it got worse”, but also explain “why it got worse”.

### 2.3 Aggregation/statistics layer: compress raw into a reviewer-friendly three-table system

Because packet-level raw data can be large, review checks are aligned to aggregated CSV tables:

- **summary_metrics**: distance-binned PDR, p50/p95/p99, late ratio, plus mean evidence fields;  
- **position_heterogeneity**: UrbMask same-distance heterogeneity table (within a fixed distance band, binned by `mid_x`);  
- **tunnel_segments**: Tunnel segmented statistics table (within a fixed distance band, binned by `tunnel_u`).

This three-table system is not “for producing more files”; it corresponds strictly to mechanism explanations: Fig05 requires the heterogeneity table; Fig06 requires the segmented table—otherwise conclusions are not auditable.

### 2.4 Figure production layer: separating “final figures” vs “transparent supporting figures”

Two categories of figures are produced:

- **Final figures (Fig01–Fig07)**: unified definitions, high information density, intended for submission/defense;  
- **Supporting figures (deliverable F1/F2/F3)**: cover all combinations (scenario × ret × seeds) for transparency and drill-down; they do not replace the final definitions.

---

## 3 Significance, value, and contributions

This section follows academic “value—contribution—verifiability” phrasing, emphasizing significance consistent with the actual work (no exaggeration).

### 3.1 Value: why this work is important and necessary

**(1) Service-aligned definitions: explicitly modeling “late” in the success definition**  
In real systems, “received but too late” is equivalent to failure for cooperative decision-making. This project explicitly distinguishes `success_phy` from `success`, and uses `late` to represent timeliness failures, so conclusions directly serve safety-service constraints rather than staying at a generic network-metric level.

**(2) Mechanizing key degraded-environment phenomena: quantifying heterogeneity and segmented effects**  
The main difficulty of urban street canyons is not just “worse on average”, but same-distance heterogeneity; the main difficulty of tunnels is not only “overall worse”, but segmented impairment and thickened tails. By `position_heterogeneity` and `tunnel_segments`, the project elevates these from intuitive descriptions to auditable statistical results.

**(3) Explainable trade-offs under congestion and enhancements: avoiding single-metric misleading**  
Retransmission gains often come with tail costs; under congestion, improved PHY success may be offset by increased late deliveries. With mechanism decomposition (Fig03) and congestion-proxy evidence (Fig04), trade-offs are grounded in data fields and figures rather than verbal discussion.

**(4) Strong reproducibility: reviewers can verify and redraw on any machine**  
The combination of run_id contract, command snapshots, aggregated tables, and final figures enables verification without depending on the author’s environment, while preserving a stable skeleton for future extensions.

### 3.2 Contributions: concrete contributions delivered by this project

**Contribution 1: standardized three-scenario construction with mechanistic interpretability**  
- RefPlus: controllable baseline (geometry + traffic + signals) for attribution;  
- UrbMaskPlus: continuous blockage intensity `d_min→b` + LOS/NLOS mixture + optional reflection-equivalent term, capturing same-distance heterogeneity;  
- TunnelPlus: segmented position `tunnel_u` + bell+fade impairment curves + tail-delay injection, capturing segmentation and tail effects.

**Contribution 2: congestion-proxy evidence chain and Cong-only ablation framework**  
- Using `n_cs`/CBR/`p_col`/`cong_delay_ms` to make explicit the pathway “traffic density → channel busy → collision/queuing → thicker tail”;  
- NoCong vs Cong differs only by a switch, ensuring attribution.

**Contribution 3: unified gain–cost evaluation with ret=0/1/2 as the main lever**  
- Not only showing PDR improvement, but also p95/p99 inflation and late risk;  
- Upgrading strategy selection from “pursue higher PDR” to “trade-off under joint reliability & timeliness constraints”.

**Contribution 4: auditable evidence-chain design (figure–table–field closure)**  
- Fig05 ⇔ position_heterogeneity ⇔ (`d_min`, `b`, `g_refl`);  
- Fig06 ⇔ tunnel_segments ⇔ (`tunnel_u`, delay tail);  
- Fig03/Fig04 ⇔ (`success_phy`, `late`, `p_col`, `cong_delay_ms`).

### 3.3 Verifiability: a minimal verification path for reviewers

A recommended “shortest verification path”:

1) Browse Fig01–Fig07 (closed-loop conclusions);  
2) Check run_commands (protocol frozen, minimal NoCong/Cong differences);  
3) Verify aggregated-table fields and definitions (tables);  
4) If drill-down is needed, consult supporting figures F1/F2/F3 and corresponding tables.

---

## 4 Three-scenario construction (detailed, with equation system)

This chapter follows academic “system model / scenario settings” writing: starting from common coordinates and time discretization, then detailing RefPlus/UrbMaskPlus/TunnelPlus, with a report-ready equation set. Note: this project is a system-level evaluation; the goal is interpretable conclusions under controllable complexity, not to replace full-stack ray tracing or full protocol-stack simulation.

### 4.0 Common conventions: coordinates, discrete time, and link midpoint

- Planar coordinates: main corridor along the \(x\)-axis, lateral offset as \(y\).  
- Discrete-time simulation with step \(\Delta t\); messages generated at frequency \(f_m\).  
- For a Tx/Rx link, define:

$$
d=\|\mathbf{p}_{tx}-\mathbf{p}_{rx}\|_2,\qquad 
\mathbf{p}_{mid}=\frac{\mathbf{p}_{tx}+\mathbf{p}_{rx}}{2},\qquad
x_{mid}=\mathbf{p}_{mid,x}.
$$

### 4.1 RefPlus: controllable baseline (geometry + IDM + signals + cross/turn)

#### 4.1.1 Geometry: corridor centerline and multi-lanes

RefPlus uses a 3 km corridor. The S-bend is described by a smooth function (example form):

$$
y_c(x)=A\sin\!\Big(2\pi\,\frac{x-x_0}{x_1-x_0}\Big)\cdot \mathbb{I}_{[x_0,x_1]}(x),
$$

where \(A\) is the lateral amplitude and \([x_0,x_1]\) is the S-bend interval. Lane centerlines can be written as:

$$
y_{\ell}^{(\pm)}(x)=y_c(x)\ \pm\ \Big(\frac{g}{2}+\big(\ell+\tfrac{1}{2}\big)w\Big),
$$

where \(g\) is the median width, \(w\) is lane width, and \(\ell\in\{0,1,2\}\) indexes the 3 lanes per direction.

#### 4.1.2 Traffic: IDM car-following (forming queues/platoons/stop-and-go waves)

$$
\dot v = a\Big[1-\Big(\frac{v}{v_0}\Big)^{\delta}-\Big(\frac{s^{\ast}(v,\Delta v)}{s}\Big)^2\Big],
\qquad
s^{\ast}(v,\Delta v)=s_0+vT+\frac{v\Delta v}{2\sqrt{ab}},
$$

where \(v\) is ego speed, \(\Delta v\) is relative speed, \(s\) is headway, \(v_0\) desired speed, \(T\) desired time gap, \(a,b\) maximum acceleration / comfortable deceleration, \(s_0\) minimum gap, and \(\delta\) is usually 4. Under signal constraints, this model naturally produces queue formation and platoon release, providing structured sources for the congestion proxy \(n_{cs}\).

#### 4.1.3 Signals and intersections: phases and offsets

Let cycle length be \(C\) and main-road green be \(G\). The release indicator can be expressed as:

$$
\phi_{main}(t)=\mathbb{I}\big(\ (t+\tau)\bmod C\in[0,G)\ \big),
$$

where \(\tau\) is an intersection offset (for phase staggering). In intersection regions, cross-flow and turn probabilities (right/left/straight) are introduced to control weaving complexity and local density peaks.

### 4.2 UrbMaskPlus: urban blockage/street canyon (buildings + continuous blockage intensity + reflection-equivalent term)

#### 4.2.1 Building geometry: a set of rectangular blocks

Represent building footprints by a rectangle set \(\mathcal{B}=\{B_k\}\), each \(B_k\) defined by \((x_{min},x_{max},y_{min},y_{max},h)\).

#### 4.2.2 Minimum distance from link segment to buildings \(d_{min}\)

$$
d_{min}=\min_{B_k\in\mathcal{B}} \ \mathrm{dist}\big(\overline{\mathbf{p}_{tx}\mathbf{p}_{rx}},\partial B_k\big)
$$

#### 4.2.3 Continuous blockage intensity \(b\)

$$
b=\exp\!\left(-\Big(\frac{d_{min}}{T}\Big)^2\right)
$$

#### 4.2.4 LOS/NLOS mixed success probability

$$
p_{succ}(d,b)=(1-b)\,p_{LOS}(d)+b\,p_{NLOS}(d)
$$

#### 4.2.5 Reflection-equivalent term (optional)

$$
G_{refl}(d_{min},b)=G_{max}\exp(-d_{min}/d_0)\,b
$$

#### 4.2.6 Same-distance heterogeneity statistics (within a fixed distance band, bin by \(x_{mid}\))

$$
\mathcal{S}_j=\{\,\text{links}: d\in[d_1,d_2],\ x_{mid}\in I_j\,\}
$$

### 4.3 TunnelPlus: segmented tunnel impairment (tunnel_u + bell+fade + tail-delay injection)

#### 4.3.1 Normalized tunnel position \(u\)

$$
u=\mathrm{clip}\Big(\frac{x_{mid}-x_0}{x_1-x_0},0,1\Big)
$$

#### 4.3.2 Impairment shape: bell + fade

$$
\mathrm{bell}(u)=\sin^2(\pi u),\qquad \mathrm{bell}_\gamma(u)=\mathrm{bell}(u)^\gamma
$$

#### 4.3.3 Tail-delay injection (illustrative form)

$$
D = D_0 + b_{tun}\cdot D_{extra},\qquad D_{extra}\sim \mathrm{Exp}(\lambda)
$$

#### 4.3.4 Segmented statistics (bin by \(u\))

$$
\mathcal{T}_j=\{\,\text{links}: d\in[d_1,d_2],\ u\in J_j\,\}
$$

### 4.4 Evidence-field design (supporting the “phenomenon–mechanism–implication” closure)

- UrbMask: `d_min`, `b`, `g_refl` ⇔ same-distance heterogeneity (Fig05)  
- Tunnel: `tunnel_u` ⇔ segmented impairment (Fig06)  
- Cong: `n_cs`, CBR, `p_col`, `cong_delay_ms` ⇔ composite failure decomposition (Fig03/04)

---

## 5 Mechanism modeling and metric system

Define \(S_{phy}\in\{0,1\}\), \(L\in\{0,1\}\), \(S=S_{phy}(1-L)\), and use:

$$
\mathrm{PDR}\approx \mathrm{PDR}_{phy}\cdot (1-\mathrm{LateRatio}_{phy})
$$

A conceptual form for retransmission and delay accumulation:

$$
\mathbb{P}(S_{phy}=1)=1-\prod_{k=0}^{ret}(1-p_k),
\qquad
D = D_{base} + D_{prop} + D_{queue} + D_{cong} + D_{retry}
$$

Congestion proxy (illustrative form):

$$
\mathrm{CBR}=\min\big(1,\ (n_{cs}-1)\lambda\tau_{air}\big),
\qquad
p_{col}=1-\exp(-k\,\mathrm{CBR}),
\qquad
D_{cong}\sim \mathrm{Exp}(\beta(\mathrm{CBR}))
$$

---

## 6 Reproducibility and audit mechanisms (rigorous phrasing for reviewers)

Let `$ROOT` denote the repository root. Outputs follow the contract `runs/<run_id>/` and include `run_commands.txt`. The difference between NoCong and Cong is constrained to the congestion switch to ensure attribution. Recommended reproduction is hierarchical: smoke → small → S20.


# Deconstruction and discussion of results (Raw-derived, S20)

## 1. Data sources and statistical pipeline (raw → cache → figures)

These results are based on two strictly controlled final experiments: **A_Final_NoCong_S20** and **A_Final_Cong_S20**. The common configuration (as recorded in final commands and project config files) is identical: scenarios are **Ref / UrbMask / Tunnel**, retransmission policies **ret=0/1/2**, random seeds **seed=1–20**, and service statistics focus on the **0–200 m** short-range region. The only difference is whether congestion contention is enabled, forming a strict ablation contrast (NoCong vs Cong).

To avoid “sparse tables leading to unsmooth/unreliable curves”, results are not plotted directly from the tables. Instead, a two-stage statistics pipeline is used:
(1) aggregate raw packet-level data by distance bins to collect counts and histogram statistics needed for quantiles;  
(2) cache the aggregated results into `.mat` (for reproducibility and fast re-plotting), then use MATLAB to uniformly render the 7 final figures.
Cache file names expose key resolutions, e.g., **distance bin=2 m**, **delay histogram bin=0.25 ms**, and the mechanism-analysis distance band **band=80–100 m** (used for UrbMask spatial heterogeneity and Tunnel segmentation analysis).

---

## 2. Metric definitions and the hierarchy of “success” (avoiding ambiguity)

With service timeliness constraints (deadlines), “success” in V2X has at least two layers: **PHY success** and **timely success**. Therefore, this report uses three complementary metrics:

1. **timely PDR (denoted as PDR or timely_rate in the figures)**  
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\frac{N*{\mathrm{success}}(d)}{N_{\mathrm{total}}(d)}
$$
where (N_{\mathrm{success}}) is the number of packets that arrive within the deadline.

2. **PHY success rate (denoted as phy_rate in the figures)**  
$$
\mathrm{PDR}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{success_phy}}(d)}{N_{\mathrm{total}}(d)}
$$
where (N_{\mathrm{success_phy}}) is the number of packets that are PHY-decoded successfully (timeliness not required).

3. **late ratio among PHY successes (denoted as late_ratio_phy in the figures)**  
$$
\mathrm{late_ratio}*{\mathrm{phy}}(d)=\frac{N*{\mathrm{late}}(d)}{N_{\mathrm{success_phy}}(d)}
$$
where (N_{\mathrm{late}}) is the number of packets that are “PHY-successful but miss the deadline”.

They satisfy the key identity (explicit in Fig03):
$$
\mathrm{PDR}*{\mathrm{timely}}(d)=\mathrm{PDR}*{\mathrm{phy}}(d)\cdot\left(1-\mathrm{late_ratio}_{\mathrm{phy}}(d)\right)
$$
This identity decomposes the causes of “timely PDR degradation” into: **(i) reduced PHY success rate** and **(ii) increased late penalty among PHY successes**, enabling a closed-loop “phenomenon–mechanism–implication” argument.

---

# Fig01–Fig07: figure-by-figure deconstruction (usable as main text + captions)

## Fig01: PDR vs distance (dist≤200 m, three-scenario contrast; NoCong solid, Cong dashed)

![](assets/final_figures_A_preview/Fig01_PDR_Focus.png)

**Meaning and role**  
Fig01 is the “main phenomenon figure” of the whole results, addressing: under degraded environments and congestion contention, how does reliability (timely PDR) decay with distance? How much gain does the retransmission policy (ret) deliver?

**Observation 1: propagation-dominated degradation and monotonic gains of ret under NoCong**  
Under NoCong, all three scenarios show the typical monotonically decreasing shape with distance. Increasing ret from 0→1→2 shifts curves upward, indicating retransmissions improve timely success probability. Fig07 provides clear numeric anchors for dist≤200 m:

- Ref: PDR = 0.439 (ret0) → 0.537 (ret1) → 0.582 (ret2)  
- UrbMask: PDR = 0.462 (ret0) → 0.584 (ret1) → 0.644 (ret2)  
- Tunnel: PDR = 0.368 (ret0) → 0.489 (ret1) → 0.542 (ret2)

This indicates: without congestion contention, the bottleneck is mainly propagation success (blockage/tunnel impairments, etc.); retransmissions can convert part of PHY failures into timely successes, yielding notable reliability gains.

**Observation 2: global collapse under Cong and “scenario differences become second-order”**  
Under Cong, PDR curves are suppressed to a much smaller magnitude, and the gaps between scenarios shrink considerably. Fig07 (dist≤200 m) shows:

- Ref: PDR = 0.054 (ret0) → 0.090 (ret1) → 0.117 (ret2)  
- UrbMask: PDR = 0.057 (ret0) → 0.097 (ret1) → 0.126 (ret2)  
- Tunnel: PDR = 0.049 (ret0) → 0.082 (ret1) → 0.107 (ret2)

This suggests: when the channel is in a busy, competitive state, the dominant bottleneck shifts from propagation to **MAC contention/collision/queuing**. Environmental differences (Ref/UrbMask/Tunnel) are “squeezed” into second-order factors in timely PDR, a judgement further supported by congestion-proxy evidence in Fig04.

**Report-ready conclusion**  
Under NoCong, ret yields stable reliability gains and clearly reveals scenario-specific propagation differences; under Cong, timely PDR collapses globally and scenario differences converge, indicating congestion contention becomes the dominant bottleneck.

---

## Fig02: Tail delays (p95/p99) vs distance (NoCong/Cong in two rows; ret=0 vs ret=2)

![](assets/final_figures_A_preview/Fig02_TailDelay_p95p99_Ret0Ret2.png)

**Meaning and role**  
Fig02 is the core “cost-dimension” figure: how large is the tail-delay cost brought by reliability enhancement (higher ret)? Does congestion push tail delays toward the deadline, thus activating timeliness penalties?

**NoCong (top row): tails increase with ret but remain far below the deadline (late≈0)**  
In the top row (NoCong), ret=0 has p95 in low millisecond range; ret=2 raises p95/p99 significantly to the 10–30 ms range, indicating retransmissions thicken the arrival-time tail of successful packets. Fig07 (dist≤200 m) provides p95 anchors:

- Ref: p95=2.9 ms (ret0) → 22.4 ms (ret2)  
- UrbMask: p95=3.9 ms (ret0) → 22.9 ms (ret2)  
- Tunnel: p95=3.9 ms (ret0) → 22.9 ms (ret2)

Meanwhile, late=0.0% for NoCong in Fig07, showing that without congestion, even though tails rise, they remain well below the deadline; timeliness penalties are not the primary failure source.

**Cong (bottom row): tails lifted to 70–100 ms; the deadline materially participates in failures**  
In the bottom row (Cong), p95 is around 70–85 ms and p99 approaches 90–100 ms; the whole tail shifts upward toward the deadline scale. Fig07 (dist≤200 m) p95 anchors:

- Ref: p95=69.6 ms (ret0) → 79.9 ms (ret2)  
- UrbMask: p95=70.1 ms (ret0) → 79.9 ms (ret2)  
- Tunnel: p95=69.1 ms (ret0) → 79.6 ms (ret2)

Hence under Cong, increasing ret may improve PDR (Fig01/Fig07) but also further elevates tail delay and introduces explicit “PHY-success but late” penalties (late% rises), which is quantitatively decomposed in Fig03.

**Report-ready conclusion**  
Under NoCong, enhancement mainly trades reliability gains against an acceptable tail-delay increase; under Cong, tails are pushed close to the deadline, making timeliness penalties a major component of reliability degradation.

---

## Fig03: Cong-only decomposition (PHY success, timely success, and late penalty)

![](assets/final_figures_A_preview/Fig03_Cong_Decomposition_3Curves.png)

**Meaning and role**  
Fig03 provides the strongest mechanistic interpretability: why is timely PDR so low under Cong? Is the loss mainly due to PHY failures or late penalties? How do the gains and costs of ret appear in this decomposition?

**How to read (must be stated in the report)**  
Each subplot draws (left axis):

- Blue: \(\mathrm{PDR}*{\mathrm{phy}}=N*{\mathrm{success_phy}}/N_{\mathrm{total}}\)  
- Green: \(\mathrm{PDR}*{\mathrm{timely}}=N*{\mathrm{success}}/N_{\mathrm{total}}\)

and (right axis) the orange dashed curve:

- \(\mathrm{late_ratio}*{\mathrm{phy}}=N*{\mathrm{late}}/N_{\mathrm{success_phy}}\)

linked by:
$$
\mathrm{PDR}*{\mathrm{timely}}=\mathrm{PDR}*{\mathrm{phy}}\cdot(1-\mathrm{late_ratio}_{\mathrm{phy}})
$$

**Observation 1: the first major loss term is low PHY success under Cong (contention/collision)**  
Within 0–200 m, the blue curve (phy_rate) itself is already low, consistent with high p_col and substantial congestion delay in Fig04, indicating the primary loss originates from contention and collision causing PHY failures. With the multiplicative structure, even moderate late ratios can still yield low timely PDR when phy_rate is small.

**Observation 2: with distance, rising late_ratio further widens the gap between timely_rate and phy_rate**  
As distance increases, the orange curve (late_ratio_phy) rises, making the green curve (timely) deviate further below the blue (PHY). This matches Fig02: congestion-induced queuing/deferral shifts the tail upward, converting part of PHY successes into late deliveries, thereby reducing timely PDR.

**Observation 3: the “double-edged” effect of ret=2 is quantitatively explained**  
Increasing ret from 0 to 2 tends to raise phy_rate (more packets eventually succeed at PHY), so timely PDR increases; meanwhile, late_ratio can rise in some distance ranges, increasing timeliness penalty. Fig07 quantifies this double-edged effect via late%:

- Ref: late=5.5% (ret0) → 8.7% (ret2)  
- UrbMask: late=5.4% (ret0) → 8.4% (ret2)  
- Tunnel: late=5.1% (ret0) → 7.9% (ret2)

Thus, under Cong, the net benefit of ret should be understood as the competition between “PHY success gain” and “late-penalty increment”, bounded by a clear multiplicative structure.

**Report-ready conclusion**  
Under Cong, timely PDR degradation can be rigorously decomposed into reduced PHY success and increased late penalty; retransmissions both improve PHY success and increase late probability, with a clear multiplicative mechanism and quantifiable boundaries.

---

## Fig04: Congestion proxy evidence (mean ± std band; evidence of consistent congestion strength)

![](assets/final_figures_A_preview/Fig04_CongProxy_MeanStdBand.png)

**Meaning and role**  
Fig04 provides mechanistic evidence: why do the three propagation environments behave similarly under Cong? Is congestion strength approximately consistent across scenarios so that congestion becomes the dominant bottleneck?

**Observation: high consistency of congestion proxies across scenarios with narrow std bands**  
The three subplots show average CBR, average p_col, and average congestion delay with mean curves and ±1 std bands across scenarios. Over dist≤200 m, curves have similar shapes and the variance bands are narrow, indicating Cong constructs a similar “busy-channel intensity” across scenarios. Under this background, timely PDR is mainly limited by collision/backoff/queuing, and propagation differences become second-order—explaining the convergence in Fig01 under Cong.

**Report-ready conclusion**  
Congestion proxies are highly consistent across the three scenarios (close means, narrow variance bands), supporting the judgement that congestion dominance drives scenario convergence.

---

## Fig05: UrbMask spatial heterogeneity (spatial heterogeneity and congestion amplification)

![](assets/final_figures_A_preview/Fig05_UrbMask_Heterogeneity_LinesAndRatio.png)

**Meaning and role**  
Fig05 shifts UrbMask analysis from distance domain to space domain: does urban blockage exhibit strong spatial heterogeneity? Does congestion amplify this heterogeneity, creating locally unavailable regions?

**Top (NoCong): PDR_band (80–100 m) shows structured variation with mid_x**  
Within the fixed distance band (80–100 m), PDR_band varies continuously with mid_x, indicating strong location dependence caused by blockage/reflection. ret=2 lifts the overall level but preserves the spatial texture, meaning retransmissions improve reliability but do not eliminate non-uniformity.

**Bottom (ratio=Cong/NoCong): congestion penalizes weak spatial regions more strongly (lower ratio)**  
The ratio is significantly below 1 across the domain, with more pronounced drops in certain mid_x intervals. Because the ratio is normalized at the same position and distance band, it indicates: congestion not only lowers average reliability, but also imposes stronger penalties on already weak-coverage / weak-reflection regions, pushing some local areas closer to unavailability. This provides direct evidence that risk under degraded environments has spatial concentration.

**Report-ready conclusion**  
UrbMask reliability degradation exhibits strong spatial heterogeneity; congestion amplifies weak regions via lower Cong/NoCong ratios, suggesting a location-dependent local amplification of risk.

---

## Fig06: Tunnel segmented effect (inside vs outside; segmented evidence and busy-hour extremes)

![](assets/final_figures_A_preview/Fig06_Tunnel_InsideOutside_Bars_UnifiedY.png)

**Meaning and role**  
Fig06 verifies the “segmented impairment mechanism” of tunnels: is inside-tunnel performance significantly worse than outside? Does congestion push the inside closer to unusable? Can ret gains and late penalties be directly quantified under inside/outside contrasts?

**NoCong (top row): inside is significantly worse than outside; ret improves both**  
Bar height is timely PDR (band 80–100 m), with bar-top labels showing PHY and late (NoCong late≈0). Typical values (from labels):

- ret0: inside phy=0.115, outside phy=0.182  
- ret1: inside phy=0.214, outside phy=0.333  
- ret2: inside phy=0.304, outside phy=0.451  

This shows the inside-tunnel propagation conditions reduce PHY success rate and structurally degrade timely PDR. Retransmissions raise PHY success and thus timely PDR, but the inside/outside gap persists, reflecting stable segmentation.

**Cong (bottom row): inside becomes near-unusable and late penalties appear; ret gains come with higher lateness risk**  
Under Cong, inside/outside bars are lower and late>0. Typical label values:

- ret0: inside phy=0.015, late=0.059; outside phy=0.034, late=0.030  
- ret2: inside phy=0.043, late=0.076; outside phy=0.097, late=0.044  

Thus under busy-hour contention, inside not only has lower PHY success, but PHY-successful packets are more likely to miss deadlines due to queuing/backoff, further reducing timely PDR. This closes the loop with Fig02 (tails near deadline) and Fig03 (rising late_ratio).

**Report-ready conclusion**  
The tunnel scenario exhibits stable inside/outside segmentation: inside is significantly worse under both NoCong and Cong; congestion pushes inside closer to unavailability and introduces notable late penalties; retransmissions improve PHY success but also increase lateness risk.

---

## Fig07: Summary matrices (dist≤200 m numeric anchors and conclusion closure)

![](assets/final_figures_A_preview/Fig07_Summary_Matrices.png)

**Meaning and role**  
Fig07 is the “closure page” of conclusions: for dist≤200 m, it provides the triplet **PDR, p95, late%** for each (scenario × ret × congestion regime). It is used to:
(1) offer citable numeric anchors;  
(2) summarize the 3D trade-off among reliability, tail delay, and timeliness penalty;  
(3) support engineering implications and boundary judgements.

**NoCong: significant reliability gains, explicit tail costs, late≈0**

- Ref: PDR 0.439→0.582; p95 2.9→22.4 ms; late 0.0%  
- UrbMask: PDR 0.462→0.644; p95 3.9→22.9 ms; late 0.0%  
- Tunnel: PDR 0.368→0.542; p95 3.9→22.9 ms; late 0.0%  

This pins down, quantitatively, the “retransmission gain vs tail-delay cost” trade-off under NoCong.

**Cong: low absolute reliability, tails near deadline, late% increases with ret**

- Ref: PDR 0.054→0.117; p95 69.6→79.9 ms; late 5.5→8.7%  
- UrbMask: PDR 0.057→0.126; p95 70.1→79.9 ms; late 5.4→8.4%  
- Tunnel: PDR 0.049→0.107; p95 69.1→79.6 ms; late 5.1→7.9%  

This shows that under busy-hour contention, the marginal benefit of retransmissions becomes much smaller, and the cost shifts from “delay increase” to the more service-critical “timeliness penalty increase”.

**Report-ready conclusion**  
Fig07 jointly quantifies reliability, tail delay, and timeliness penalty: under NoCong, retransmissions significantly improve reliability at the cost of higher tail delay; under Cong, absolute reliability is suppressed by congestion dominance and retransmissions increase late%, implying enhancement strategies should be co-designed with congestion control.

---

# Cross-figure synthesis (journal-style “phenomenon–mechanism–implication” closure)

Combining main results (Fig01/02/07) with mechanistic evidence (Fig03/04/05/06), we obtain the following closed-loop conclusions:

1. **Bottleneck shift**:  
   Under NoCong, the bottleneck is mainly propagation impairment (scenario differences are clear); under Cong, the bottleneck shifts to contention/collision/queuing, causing global PDR collapse and making scenario differences second-order (Fig01 + Fig04).

2. **Gain–cost boundary**:  
   Under NoCong, retransmissions provide significant reliability gains with quantifiable tail costs (Fig02 + Fig07); under Cong, retransmissions still help but substantially increase late% (timeliness penalty), partially offsetting the net gain in timely reliability (Fig03 + Fig07).

3. **Structured degradation**:  
   Urban blockage does not only reduce averages; it introduces spatial heterogeneity, and congestion amplifies degradation in weak regions (Fig05). Tunnel impairment has clear inside/outside segmentation, and busy hours push inside closer to unusable (Fig06).

4. **Engineering implication**:  
   In real systems where degraded environments and busy-hour contention coexist, retransmissions alone are insufficient to jointly optimize reliability and timeliness. Retransmissions should be jointly designed with congestion control (rate limiting, scheduling, prioritization, resource allocation, etc.) so that both phy_rate and late_ratio are controlled, thereby improving timely PDR (Fig03’s decomposition directly provides optimization targets).

---

# Limitations and recommended statements for rigor

To avoid reviewers “picking details”, the following statements are recommended (copy-ready):

1. **Statistical resolution and smoothing**: curves are based on distance-bin aggregation and mild smoothing (for visualization only), without changing raw counts and histogram statistics; key numeric conclusions rely on weighted summaries in Fig06/Fig07.  
2. **Quantile estimation**: p95/p99 are estimated from delay histograms of timely successes; when successful samples in a distance bin are insufficient, quantiles are uncertain—thus the plotting uses masking/screening to avoid unreliable estimates affecting conclusions.  
3. **Role of mechanism proxies**: congestion proxies in Fig04 are controllable internal model indicators used to support the evidence chain of “congestion dominance”, rather than a bit-level reproduction of a specific standardized protocol stack; their value lies in enabling explainable contrasts and strategy-boundary discussions.


## Code structure overview (quick navigation)

Project code is under `03_sim_A/py/` and `03_sim_A/py/modules/`. The overall pipeline is orchestrated by `run_pipeline_A.py`: input generation → simulation → aggregation → plotting → audit snapshots.

A more detailed “file-by-file guide (key functions/classes and common parameter entry points)” is provided in Notebook 2.
